# Author topics on both network backbones

This notebook and the command-line workflow use the same implementation in `src/author_analysis/`.
Each network selects its authors; all post types, including retweets, contribute to author documents.
The default minimum is three nonempty cleaned posts. Model selection and final fitting are separate
for each backbone. See the source README for settings, input validation and downstream figures.


In [ ]:
import os
import sys
from pathlib import Path

RUNNING_LOCALLY = "google.colab" not in sys.modules
if RUNNING_LOCALLY:
    DATA = Path(os.environ["TWITTER_AI_DATA_ROOT"])
    OUT = Path(os.environ["TWITTER_AI_OUTPUT_ROOT"])
    REPO = Path.cwd().resolve()
    while not (REPO / "src/author_analysis/run.py").is_file() and REPO != REPO.parent:
        REPO = REPO.parent
    assert (REPO / "src/author_analysis/run.py").is_file(), "Open inside the repository checkout"
else:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA = Path("/content/drive/MyDrive/Colab Projects/AI Public Trust")
    OUT = DATA / "Models/Topic Modeling/LDA/backbone_analysis"
    REPO = Path("/content/twitter_ai")


In [ ]:
import subprocess
if not RUNNING_LOCALLY and not REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/IgnacioOQ/twitter_ai.git", str(REPO)], check=True)


In [ ]:
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))


In [ ]:
if not RUNNING_LOCALLY:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r",
                    str(REPO / "src/author_analysis/requirements.txt")], check=True)


In [ ]:
from src.author_analysis.run import main as run_stage
from src.author_analysis.topic_model import ModelSettings

BACKBONES = ["RetweetedOnce", "LWCC"]
SMOKE_TEST = True  # switch off for the full run; smoke outputs stay in OUT/smoke/
MODEL_CONFIG = None  # optional JSON settings; defaults preserve the backbone model-selection method
JOBS = int(os.environ.get("SLURM_CPUS_PER_TASK", "1"))
print(ModelSettings.load(MODEL_CONFIG, n_jobs=JOBS, smoke=SMOKE_TEST))

def stage(name, backbone):
    args = [name, "--data-root", str(DATA), "--output-root", str(OUT),
            "--backbone", backbone, "--jobs", str(JOBS)]
    if SMOKE_TEST: args += ["--smoke"]
    if MODEL_CONFIG: args += ["--model-config", str(MODEL_CONFIG)]
    if run_stage(args) != 0: raise RuntimeError(f"{backbone}: {name} failed")


## Prepare documents and fit topics

Defaults: TF-IDF unigram/bigram representations; K=8,12,16,20; alpha=0.05,0.1,0.3;
eta=0.01,0.1; 100,000-author selection sample; 20,000 coherence documents; 50 iterations.
The same coherence/diversity/collapse/evenness ranking selects a model, then refits it on all
eligible authors of that backbone. Topic labels must be assigned to the resulting model.


In [ ]:
for backbone in BACKBONES:
    stage("prepare", backbone)
    stage("topics", backbone)


## Inspect the selected models

Results remain external to the repository. Grid tables and top terms show what was selected.
Topic numbers in separate backbone models are not equivalent labels.


In [ ]:
import json
import pandas as pd
ROOT = OUT / "smoke" if SMOKE_TEST else OUT
for backbone in BACKBONES:
    topic_dir = ROOT / "networks" / backbone / "topics"
    print(backbone, json.loads((topic_dir / "model.json").read_text()))
    display(pd.read_csv(topic_dir / "grid_results.csv").head(10))
    display(pd.read_csv(topic_dir / "top_terms.csv"))


In [ ]:
import matplotlib.pyplot as plt
import umap

fig, axes = plt.subplots(len(BACKBONES), 4, figsize=(20, 4*len(BACKBONES)), squeeze=False)
for row, backbone in zip(axes, BACKBONES):
    topic_dir = ROOT / "networks" / backbone / "topics"
    grid = pd.read_csv(topic_dir / "grid_results.csv")
    for ax, metric in zip(row, ["coherence_c_v", "diversity", "largest_topic_share", "combined_score"]):
        for representation, sub in grid.groupby("representation"):
            grouped = sub.groupby("K")[metric]
            best = grouped.min() if metric == "largest_topic_share" else grouped.max()
            ax.plot(best.index, best.values, marker="o", label=representation)
        ax.set_title(f"{backbone}: {metric}")
        ax.set_xlabel("K")
    row[0].legend()
fig.tight_layout()
fig.savefig(ROOT / "grid_quality_vs_K.png", dpi=150)
plt.show()

for backbone in BACKBONES:
    topic_dir = ROOT / "networks" / backbone / "topics"
    table = pd.read_csv(topic_dir / "author_topics.csv", dtype={"author_id": str})
    info = json.loads((topic_dir / "model.json").read_text())
    columns = [f"topic_{i}" for i in range(info["K"])]
    sample = table.sample(min(5_000 if SMOKE_TEST else 50_000, len(table)), random_state=42)
    if len(sample) < 4: continue
    points = umap.UMAP(n_neighbors=min(30,len(sample)-1), min_dist=.1,
                       metric="cosine", random_state=42).fit_transform(sample[columns].values)
    fig, axes = plt.subplots(1,2,figsize=(18,7))
    sizes = table.dominant_topic.value_counts().reindex(range(info["K"]), fill_value=0)
    axes[0].bar(sizes.index, sizes.values)
    axes[0].set_title(f"{backbone}: dominant topics")
    axes[0].set_ylabel("authors")
    points_plot = axes[1].scatter(points[:,0],points[:,1],c=sample.dominant_topic,cmap="tab20",s=2,alpha=.5)
    axes[1].set_title(f"{backbone}: sampled topic-mixture UMAP")
    fig.colorbar(points_plot,ax=axes[1],label="topic")
    fig.savefig(topic_dir / "topic_sizes_and_umap.png",dpi=150)
    plt.show()


## Affect and community analysis

Run after the classified sentiment and emotion files are repaired and verified.
Both average all post types for topic-eligible authors. Missing or mismatched posts fail explicitly.
Weekly inference and visualisation use these same model bundles and matched IDs; see the source README.


In [ ]:
for backbone in BACKBONES:
    for name in ["affect", "match", "summarize", "communities"]:
        stage(name, backbone)
